# Поколоночный EDA рецидивов ДКА

Ноутбук показывает разведочный анализ по очищенным данным (`data/processed/dka_clean.parquet`). Тяжелая логика лежит в пакете `src` (cleaning, stats, columns, eda), здесь только представление. Методы описания и тесты выбираются по протоколу проекта из `src/stats.py`.

In [ ]:
import sys, pathlib

# Добавляем корень проекта в путь, чтобы импортировать пакет src.
p = pathlib.Path.cwd()
while not (p / 'src').exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import pandas as pd
from IPython.display import Image, display

from src import io, columns, stats, eda

eda.setup_style()
df = io.load_processed()
roles = columns.classify(df)
target = roles['target']
print('Наблюдений:', len(df), 'колонок:', df.shape[1])

## Целевая переменная: баланс классов

In [ ]:
stats.describe_categorical(df[target])

## Количественные показатели

Для каждого: описание по протоколу (M, SD и 95% ДИ при нормальности, иначе Me и квартили), критерий нормальности, сравнение групп рецидива (Стьюдент, Манна-Уитни или Бруннера-Мюнцеля).

In [ ]:
for col in roles['quantitative']:
    info = eda.eda_quantitative(df, col, target)
    print('=' * 80)
    print(col)
    for k, v in info.items():
        if k != 'фигура':
            print(f'  {k}: {v}')
    display(Image(str(eda.config.FIGURES_DIR / info['фигура'])))

## Категориальные показатели

Доли с 95% ДИ Клоппера-Пирсона, связь с целевой (хи-квадрат Пирсона или точный критерий Фишера), отношение шансов для четырехпольных таблиц.

In [ ]:
for col in roles['categorical']:
    info = eda.eda_categorical(df, col, target)
    print('=' * 80)
    print(col)
    display(stats.describe_categorical(df[col]))
    for k, v in info.items():
        if k not in ('фигура', 'категории'):
            print(f'  {k}: {v}')
    display(Image(str(eda.config.FIGURES_DIR / info['фигура'])))

## Мультиколлинеарность

Корреляции Спирмена и VIF (грубая оценка на медианно заполненных данных, с константой). VIF выше 10 указывает на сильную мультиколлинеарность, разбираем такие признаки перед обучением.

In [ ]:
vif = eda.correlation_and_vif(df, roles['quantitative'])
display(Image(str(eda.config.FIGURES_DIR / 'correlation_spearman.png')))
vif